In [1]:
import os
os.chdir('/home/smallyan/eval_agent')
print(f"Working directory: {os.getcwd()}")

Working directory: /home/smallyan/eval_agent


# Generalization Evaluation — Read-Only (No Execution)

## Overview

This evaluation assesses whether the **findings claimed in the repository** for "Linearity of Relation Decoding in Transformer Language Models" are justified as **generalizable beyond the original experimental setting**, based **solely on explicit arguments, assumptions, and evidence available by static inspection**.

**Evaluation Mode**: Read-Only Generalization Evaluation (No Execution)

**Repository Path**: `/net/scratch2/smallyan/relations_eval`

**Materials Inspected**:
- `plan.md` — Experimental plan with hypothesis, methodology, and results
- `CodeWalkthrough.md` — Code walkthrough and setup instructions
- `src/` — Source code for LRE operators, benchmarks, models, and metrics
- `demo/` — Demo notebooks for LRE and attribute lens
- `data/` — Dataset with 47 relations across factual, commonsense, linguistic, and bias categories
- `documentation.pdf` — Full documentation (referenced via plan.md excerpts)

---

## GT1: Generalization to a New Model

### Evidence from Repository Inspection

**1. Multiple Models Supported in Code (`src/models.py:22-32`)**

The codebase explicitly supports multiple models:
```python
GPT_J_NAME_SHORT = "gptj"
GPT_J_NAME = "EleutherAI/gpt-j-6B"
GPT_NEO_X_NAME_SHORT = "neox"
GPT_NEO_X_NAME = "EleutherAI/gpt-neox-20b"
LLAMA_13B_NAME = "llama-13b"
LLAMA_30B_NAME = "llama-30b"
```

**2. Model-Agnostic Architecture Handling (`src/models.py:118-202`)**

The code provides model-agnostic layer path determination:
```python
def determine_layer_paths(model, layers=None):
    # Handles GPTNeoXForCausalLM, LlamaForCausalLM, GPT-J, GPT-2
    if isinstance(model, transformers.GPTNeoXForCausalLM):
        layer_path = f"gpt_neox.layers.{layer_index}"
    elif isinstance(model, transformers.LlamaForCausalLM):
        layer_path = f"model.layers.{layer_index}"
    else:
        layer_path = f"transformer.h.{layer_index}"
```

**3. Cross-Model Results in Plan (`plan.md:43-46`)**

The plan explicitly documents cross-model analysis:
> "Cross-Model Analysis: What varied: Language models (GPT-J, GPT-2-XL, LLaMA-13B)... Main result: LRE performance strongly correlated across models (GPT-J vs GPT-2-XL: R=0.85; GPT-J vs LLaMA-13B: R=0.71); similar patterns of which relations are linearly decodable across different model architectures and sizes."

**4. Mechanistic Reasoning in Plan (`plan.md:7-9`)**

The methodology is grounded in fundamental transformer properties:
> "For each relation, the decoding procedure is approximately affine (linear relational embedding), expressed as LRE(s) = Wrs + br mapping subject s to object o. These affine transformations can be computed directly from the LM Jacobian on a prompt expressing the relation (∂o/∂s)."

This Jacobian-based approach is applicable to any differentiable transformer architecture.

### Assessment

**PASS** — The repository provides:
1. **Empirical evidence** of cross-model testing (GPT-J, GPT-2-XL, LLaMA-13B) with quantified correlation (R=0.71-0.85) documented in `plan.md`
2. **Architectural reasoning** — The LRE method is based on Jacobian computation (∂F/∂s), which is model-agnostic for differentiable transformers
3. **Code implementation** supporting multiple model architectures (`src/models.py`)

---

## GT2: Generalization to New Data

### Evidence from Repository Inspection

**1. Explicit Train/Test Split Methodology (`src/data.py:84-141`)**

The code implements systematic leave-out evaluation:
```python
def split(self, train_size: int, test_size: int | None = None):
    """Break into a train/test split."""
    # Shuffle once up front, because we're sometimes sorted
    samples = self.samples.copy()
    random.shuffle(samples)
    # Returns train and test Relation objects with distinct samples
```

**2. Few-Shot Generalization Design (`src/operators.py:211-284`)**

The `JacobianIclMeanEstimator` explicitly computes LRE from n training samples and applies to new subjects:
```python
def __call__(self, relation: data.Relation) -> LinearRelationOperator:
    samples = relation.samples
    # Estimates W and b from training samples
    for sample in samples:
        approx = functional.order_1_approx(...)
        approxes.append(approx)
    weight = torch.stack([approx.weight for approx in approxes]).mean(dim=0)
    bias = torch.stack([approx.bias for approx in approxes]).mean(dim=0)
```

**3. Multiple Trial Evaluation (`src/benchmarks.py:71-91`)**

The faithfulness benchmark explicitly tests on held-out data with multiple trials:
```python
def faithfulness(..., n_trials: int = 3, n_train: int = 5, ...):
    # For each trial, split into train/test
    for _ in range(n_trials):
        train, test = relation.split(n_train)
        # Evaluate on test samples distinct from training
```

**4. Explicit Assumptions in Plan (`plan.md:12`)**

The methodology states explicit assumptions about data:
> "Extract Linear Relational Embeddings (LREs) by computing the mean Jacobian W and bias b from n=8 examples using first-order Taylor approximation: W = E[∂F/∂s] and b = E[F(s,c) - (∂F/∂s)s], scaled by β to correct underestimation."

This first-order Taylor approximation assumes near-linear behavior, which is explicitly tested.

**5. Scope Limitations Documented (`plan.md:21`)**

Clear boundaries for when the method fails:
> "some relations like Company CEO showed <6% faithfulness indicating non-linear decoding"

**6. Zero-Shot Prompt Testing (`plan.md:38-41`)**

Attribute lens results show generalization to zero-shot prompts:
> "on distracted prompts where LM predicts wrong answer (2-3% R@1), attribute lens recovered correct fact 54-63% R@1"

### Assessment

**PASS** — The repository provides:
1. **Explicit train/test separation** in code (`src/data.py`, `src/benchmarks.py`)
2. **Stated assumptions** about data — first-order Taylor approximation with n samples
3. **Documented scope limitations** — identifies failure cases (Company CEO, large-range relations)
4. **Generalization testing** — tests on held-out samples and zero-shot prompts

---

## GT3: Method / Specificity Generalizability

### Does the Work Propose a New Method?

**YES** — The repository proposes a new method: **Linear Relational Embeddings (LRE)** estimated via **Jacobian approximation** of transformer computation.

From `plan.md`:
> "For each relation, the decoding procedure is approximately affine (linear relational embedding), expressed as LRE(s) = Wrs + br mapping subject s to object o."

### Evidence from Repository Inspection

**1. Method Applied Across Multiple Relation Types (`data/` structure)**

The dataset spans four distinct categories:
```
data/
├── bias/        (8 relations)
├── commonsense/ (9 relations)
├── factual/     (27 relations)
└── linguistic/  (7 relations)
```

**2. Multiple Estimator Variants (`src/operators.py`)**

The code implements multiple variants of the LRE method:
- `JacobianEstimator` (lines 131-180): Single-example Jacobian
- `JacobianIclMeanEstimator` (lines 210-284): Mean Jacobian with ICL
- `JacobianIclMeanEstimator_Imaginary` (lines 287-422): Experimental variant
- `Word2VecIclEstimator` (lines 545-623): Translation baseline
- `LearnedLinearEstimator` (lines 626-727): Learned linear regression

**3. Method Extended to Novel Application — Attribute Lens (`src/attributelens/`)**

The LRE method is extended to a distinct application:
```
src/attributelens/
├── __init__.py
├── attributelens.py
└── utils.py
```

From `plan.md:38-41`:
> "Attribute lens revealed latent knowledge even when LM outputs falsehoods; on distracted prompts where LM predicts wrong answer (2-3% R@1), attribute lens recovered correct fact 54-63% R@1."

**4. Explicit Method Properties Required (`plan.md:6-9`)**

The plan specifies required conditions:
1. Extractable subject representation at intermediate layers
2. Near-linear relation decoding F(s,c) ≈ Ws + b
3. Computable Jacobian ∂F/∂s

**5. Applicability Across Relation Categories (`plan.md:18-21`)**

Results documented for all relation types:
> "LRE Faithfulness Evaluation... Relations (47 total across factual, commonsense, linguistic, and bias categories)... 48% of relations achieved >60% faithfulness on GPT-J; LRE outperformed baselines (Identity, Translation, Linear Regression) across all relation types"

**6. Failure Conditions Identified (`plan.md:21`)**

The method's limitations are explicitly stated:
> "Not all relations are linearly decodable; some relations are reliably predicted but do not exhibit linear relational embeddings."

### Assessment

**PASS** — The repository:
1. **Proposes a new method** (Jacobian-based LRE estimation)
2. **Tests across multiple relation types** (47 relations in 4 categories)
3. **Extends to a novel application** (attribute lens for probing latent knowledge)
4. **Specifies required properties** (differentiable transformer, intermediate layer access, near-linear behavior)
5. **Documents failure modes** (non-linear relations, large output ranges)

---

## Binary Checklist Summary

| Criterion | Result | Evidence Location | Summary |
|-----------|--------|-------------------|---------|
| **GT1: Model Generalization** | **PASS** | `plan.md:43-46`, `src/models.py:22-32, 118-202` | Cross-model testing on GPT-J, GPT-2-XL, LLaMA-13B with R=0.71-0.85 correlation. Jacobian-based method is architecturally model-agnostic for differentiable transformers. |
| **GT2: Data Generalization** | **PASS** | `plan.md:12, 21`, `src/data.py:84-141`, `src/benchmarks.py:71-91` | Explicit train/test splits, n=8 example estimation with held-out evaluation, stated Taylor approximation assumption, documented scope limitations (non-linear relations fail). |
| **GT3: Method Generalization** | **PASS** | `plan.md:6-9, 18-21, 38-41`, `src/operators.py`, `src/attributelens/` | New LRE method tested on 47 relations across 4 categories, extended to attribute lens application, required properties and failure modes explicitly documented. |

---

## Overall Generalizability Assessment

### Summary

The repository provides **strong explicit justification** for generalizability across all three dimensions:

**Strengths:**

1. **Model Generalization (GT1)**: The repository documents empirical cross-model testing on three architecturally distinct models (GPT-2-XL, GPT-J, LLaMA-13B) with quantified correlation metrics. The Jacobian-based LRE method is grounded in fundamental transformer properties (differentiability, layer-wise computation) rather than model-specific assumptions.

2. **Data Generalization (GT2)**: The methodology explicitly separates training (n samples for LRE estimation) from evaluation (held-out subjects). The plan documents 24-trial evaluations with random sampling. Assumptions (first-order Taylor approximation, zero-mean noise) are stated, and scope limitations (non-linear relations, large output ranges) are clearly identified.

3. **Method Generalization (GT3)**: The proposed LRE estimation method is tested across 47 relations spanning factual, commonsense, linguistic, and bias categories. The method is extended to a novel probing application (attribute lens). Required properties (differentiability, intermediate layer access, near-linear behavior) and failure modes are documented.

**Key Evidence:**
- `plan.md` provides detailed experimental results with cross-model correlation (R=0.71-0.85)
- `src/models.py` implements model-agnostic layer handling
- `src/data.py` and `src/benchmarks.py` implement systematic train/test evaluation
- `src/operators.py` implements multiple LRE estimation variants
- `src/attributelens/` demonstrates method extension to novel applications

### Conclusion

All generalizability claims in the repository are **explicitly justified** through:
1. Empirical cross-validation across multiple conditions
2. Architectural/mechanistic reasoning tied to transformer properties
3. Clear documentation of assumptions and scope limitations

The evaluation does **not require execution** to verify these claims, as they are based on explicit statements, code structure, and documented results in the repository materials.

In [2]:
import json
import os

# Create the output directory
output_dir = "/net/scratch2/smallyan/relations_eval/no_exe_evaluation"
os.makedirs(output_dir, exist_ok=True)

# Create the JSON summary
summary = {
    "Checklist": {
        "GT1_ModelGeneralization": "PASS",
        "GT2_DataGeneralization": "PASS",
        "GT3_MethodGeneralization": "PASS"
    },
    "Rationale": {
        "GT1_ModelGeneralization": "The repository provides explicit empirical evidence of cross-model testing on GPT-J, GPT-2-XL, and LLaMA-13B with quantified correlation (R=0.71-0.85) documented in plan.md:43-46. The Jacobian-based LRE method (∂F/∂s) is architecturally model-agnostic for differentiable transformers. Code in src/models.py:22-32 and 118-202 implements model-agnostic layer path handling for multiple architectures (GPT-J, GPT-2-XL, GPT-NeoX, LLaMA).",
        
        "GT2_DataGeneralization": "The repository explicitly implements train/test separation (src/data.py:84-141, src/benchmarks.py:71-91) with n=8 examples for LRE estimation and evaluation on held-out subjects. Assumptions are stated in plan.md:12 (first-order Taylor approximation: W = E[∂F/∂s]). Scope limitations are documented (plan.md:21): 'some relations like Company CEO showed <6% faithfulness indicating non-linear decoding.' Zero-shot generalization is tested via attribute lens experiments.",
        
        "GT3_MethodGeneralization": "The repository proposes a new Jacobian-based LRE estimation method and tests it across 47 relations in 4 categories (data/bias, data/commonsense, data/factual, data/linguistic). The method is extended to a novel application (attribute lens, src/attributelens/) for probing latent knowledge. Required properties are specified in plan.md:6-9: subject representation extraction, near-linear F(s,c), computable Jacobian. Failure modes are documented for relations with non-linear decoding or large output ranges."
    }
}

# Save to JSON file
json_path = os.path.join(output_dir, "generalization_eval_summary.json")
with open(json_path, 'w') as f:
    json.dump(summary, f, indent=2)

print(f"JSON summary saved to: {json_path}")
print("\nJSON Content:")
print(json.dumps(summary, indent=2))

JSON summary saved to: /net/scratch2/smallyan/relations_eval/no_exe_evaluation/generalization_eval_summary.json

JSON Content:
{
  "Checklist": {
    "GT1_ModelGeneralization": "PASS",
    "GT2_DataGeneralization": "PASS",
    "GT3_MethodGeneralization": "PASS"
  },
  "Rationale": {
    "GT1_ModelGeneralization": "The repository provides explicit empirical evidence of cross-model testing on GPT-J, GPT-2-XL, and LLaMA-13B with quantified correlation (R=0.71-0.85) documented in plan.md:43-46. The Jacobian-based LRE method (\u2202F/\u2202s) is architecturally model-agnostic for differentiable transformers. Code in src/models.py:22-32 and 118-202 implements model-agnostic layer path handling for multiple architectures (GPT-J, GPT-2-XL, GPT-NeoX, LLaMA).",
    "GT2_DataGeneralization": "The repository explicitly implements train/test separation (src/data.py:84-141, src/benchmarks.py:71-91) with n=8 examples for LRE estimation and evaluation on held-out subjects. Assumptions are stated in pl

In [3]:
import shutil

# Copy the notebook to the output location
current_notebook = "/home/smallyan/eval_agent/notebooks/2026-01-15-12-43_GeneralizationEval.ipynb"
target_notebook = "/net/scratch2/smallyan/relations_eval/no_exe_evaluation/generalization_eval.ipynb"

shutil.copy(current_notebook, target_notebook)
print(f"Notebook copied to: {target_notebook}")

Notebook copied to: /net/scratch2/smallyan/relations_eval/no_exe_evaluation/generalization_eval.ipynb


In [4]:
# Verify the output files
import os

output_dir = "/net/scratch2/smallyan/relations_eval/no_exe_evaluation"
files = os.listdir(output_dir)
print(f"Files in {output_dir}:")
for f in files:
    filepath = os.path.join(output_dir, f)
    size = os.path.getsize(filepath)
    print(f"  {f}: {size} bytes")

Files in /net/scratch2/smallyan/relations_eval/no_exe_evaluation:
  generalization_eval_summary.json: 1696 bytes
  replications: 4 bytes
  generalization_eval.ipynb: 21146 bytes


In [5]:
# Print the final summary
print("=" * 70)
print("GENERALIZATION EVALUATION COMPLETE")
print("=" * 70)
print()
print("Output files created:")
print(f"  1. Notebook: {target_notebook}")
print(f"  2. JSON Summary: /net/scratch2/smallyan/relations_eval/no_exe_evaluation/generalization_eval_summary.json")
print()
print("=" * 70)
print("SUMMARY OF RESULTS")
print("=" * 70)
print()
print("| Criterion                    | Result |")
print("|------------------------------|--------|")
print("| GT1: Model Generalization    | PASS   |")
print("| GT2: Data Generalization     | PASS   |")
print("| GT3: Method Generalization   | PASS   |")
print()
print("All generalizability criteria are satisfied based on explicit")
print("arguments, assumptions, and evidence in the repository.")
print("=" * 70)

GENERALIZATION EVALUATION COMPLETE

Output files created:
  1. Notebook: /net/scratch2/smallyan/relations_eval/no_exe_evaluation/generalization_eval.ipynb
  2. JSON Summary: /net/scratch2/smallyan/relations_eval/no_exe_evaluation/generalization_eval_summary.json

SUMMARY OF RESULTS

| Criterion                    | Result |
|------------------------------|--------|
| GT1: Model Generalization    | PASS   |
| GT2: Data Generalization     | PASS   |
| GT3: Method Generalization   | PASS   |

All generalizability criteria are satisfied based on explicit
arguments, assumptions, and evidence in the repository.
